<table><tr>
<td width="76"><div align="center" style="font-size:44px">🔍</div></td>
<td><h1 style="margin:0">LAB 1 · ¿Qué me han mandado?</h1>
<b>Máster en Cyber Threat Intelligence · Módulo 06 · Sesión 44 — Análisis estático de malware</b><br>
⏱️ <b>15 minutos</b> &nbsp;·&nbsp; 🎯 Saber qué es de verdad un fichero, sin abrirlo y sin ejecutarlo</td>
</tr></table>

---

## La situación

Eres analista en el CERT de una empresa. Esta mañana han llegado **9 ficheros** desde distintos
buzones de correo. Nadie los ha abierto todavía. Tu jefa te pide una sola cosa:

> *"Dime cuáles de estos son peligrosos, y dímelo **antes** de comer."*

No puedes ejecutarlos (sería una locura). No puedes abrirlos (sería otra locura).
Solo puedes **mirarlos por fuera**. Eso es el análisis estático.

### Lo que tienes que entregar al final (apúntalo, lo pondremos en común)
1. 🏷️ **¿Qué fichero miente sobre lo que es?** (su extensión dice una cosa y por dentro es otra)
2. 🔢 **El SHA-256 de ese fichero mentiroso** y qué familia de malware dice VirusTotal que es
3. 🎁 *(Reto)* **¿Cuál de los ejecutables está "empaquetado"** y cómo lo has sabido?

---
## 🔧 Preparación · ejecuta esta celda SIEMPRE

**Cada laboratorio es un cuaderno distinto y arranca en una máquina nueva.**
Aunque vengas del laboratorio anterior, aquí no hay nada instalado y las muestras
todavía no están. No se comparte nada entre cuadernos.

Pulsa ▶️ en la celda de abajo y espera unos **40 segundos**. Solo hay que hacerlo
una vez por laboratorio.

> 📦 La segunda celda, **PLAN B**, solo hace falta si la primera no consigue las
> muestras. Si la primera termina con el listado de ficheros, ignórala y sigue.


In [ ]:
#@title ▶️ EJECUTA ESTA CELDA (botón ▶ a la izquierda) y espera ~40 segundos { display-mode: "form" }

#@markdown ---
#@markdown **No hace falta que entiendas este código todavía.** Solo prepara el laboratorio:
#@markdown instala las herramientas, descarga las muestras y las descomprime.
#@markdown ---

SAMPLES_URL = "https://github.com/jstnk9/kschool_ejercicios/raw/main/analisis_estatico/muestras_kschool.zip" #@param {type:"string"}
PASSWORD    = "infected" #@param {type:"string"}

import os, glob, subprocess

def _sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

print("1/4  Instalando herramientas de análisis...")
_sh("pip install -q pyzipper pefile oletools yara-python py-tlsh")
print("     ✔ pyzipper, pefile, oletools, yara-python, tlsh")

print("2/4  Consiguiendo las muestras...")
DEST = "/content/muestras_kschool.zip"

if not SAMPLES_URL.strip():
    # Sin URL configurada: las muestras se suben a mano con la celda de abajo.
    print("     ℹ  Este cuaderno no trae URL de descarga.")
    print("        Ve a la celda de abajo, 'PLAN B', y sube el fichero")
    print("        muestras_kschool.zip que te ha pasado el profesor.")
    print("        Es un paso normal: tarda 10 segundos.")
    ok_zip = False
else:
    url = SAMPLES_URL.strip()

    # GitHub sirve DOS urls distintas para el mismo fichero:
    #   .../blob/...  -> la pagina web que lo muestra  (HTML)
    #   .../raw/...   -> el fichero de verdad
    # Si te has copiado la de la barra del navegador, la arreglamos aqui.
    if "github.com" in url and "/blob/" in url:
        url = url.replace("/blob/", "/raw/")
        print("     ℹ  URL de GitHub corregida: /blob/ -> /raw/")
    url = url.replace("?raw=true", "").replace("?raw=1", "")

    if os.path.exists(DEST):
        os.remove(DEST)      # por si un intento anterior dejo un fichero malo

    if "drive.google.com" in url:
        _sh("pip install -q gdown")
        _sh("gdown --fuzzy '" + url + "' -O " + DEST)
    else:
        _sh("wget -q --no-check-certificate '" + url + "' -O " + DEST)

    # Comprobamos que lo descargado es DE VERDAD un ZIP mirando sus primeros
    # bytes. Que es, mira tu por donde, justo lo que vas a aprender hoy:
    # un ZIP siempre empieza por 50 4B 03 04, o sea "PK".
    cabecera = open(DEST, "rb").read(4) if os.path.exists(DEST) else b""
    ok_zip = cabecera == b"PK\x03\x04"

    if ok_zip:
        print("     ✔ Descargado (" + str(os.path.getsize(DEST)//1024) + " KB, empieza por 'PK' ✔)")
    else:
        print("     ✖ Lo que he descargado NO es un ZIP.")
        print("        Empieza por los bytes " + (cabecera.hex() or "(nada)") +
              " y un ZIP empieza siempre por 504b0304.")
        if b"<" in cabecera or b"\n" in cabecera:
            print("")
            print("        Parece una pagina HTML. Lo tipico: la URL apunta a la PAGINA")
            print("        de GitHub y no al fichero. Fijate en la diferencia:")
            print("           .../blob/main/...  <- pagina web   ✖")
            print("           .../raw/main/...   <- el fichero   ✔")
            print("        Pulsa el boton 'Raw' en GitHub y copia esa URL.")
        print("")
        print("        Alternativa: usa la celda de abajo, 'PLAN B'.")

print("3/4  Descomprimiendo (contraseña: infected)...")
import pyzipper
if ok_zip:
    try:
        with pyzipper.AESZipFile(DEST) as z:
            z.setpassword(PASSWORD.encode())
            z.extractall("/content/")
        print("     ✔ Descomprimido en /content/muestras/")
    except Exception as e:
        print("     ✖ Error al descomprimir:", e)

print("4/4  Comprobando el laboratorio...")
MUESTRAS = "/content/muestras"
ficheros = sorted(f for f in glob.glob(MUESTRAS + "/*") if not f.endswith("LEEME.txt"))
if len(ficheros) >= 9:
    print("     ✔ " + str(len(ficheros)) + " muestras listas")
    print("")
    print("=" * 52)
    print("   LABORATORIO LISTO  ✅")
    print("=" * 52)
    for f in ficheros:
        print("   {:>9,} bytes   {}".format(os.path.getsize(f), os.path.basename(f)))
else:
    print("     ✖ Solo encuentro " + str(len(ficheros)) + " ficheros. Usa la celda 'PLAN B' de abajo.")

print("""
⚠️  RECUERDA: esto son muestras REALES de malware.
    Estás dentro de una máquina virtual de Google que se destruye al cerrar.
    NO descargues estos ficheros a tu ordenador. NO los ejecutes.
    Hoy solo vamos a MIRARLOS, que es justo de lo que va el análisis estático.
""")

In [ ]:
#@title 📦 PLAN B — solo si la celda de arriba no ha conseguido las muestras { display-mode: "form" }
import glob, os

# Esta celda se puede ejecutar sola, sin haber pasado por la de arriba,
# asi que se instala ella misma lo que necesita.
try:
    import pyzipper
except ImportError:
    print("Instalando pyzipper (5 segundos)...")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyzipper"], check=False)
    import pyzipper

ZIP_YA_SUBIDO = "/content/muestras_kschool.zip"

if os.path.exists(ZIP_YA_SUBIDO):
    # ya lo habias subido con el panel de Archivos de la izquierda
    print("Encontrado", ZIP_YA_SUBIDO, "- no hace falta que lo subas otra vez.")
    nombres = [ZIP_YA_SUBIDO]
else:
    from google.colab import files
    print("Pulsa en 'Elegir archivos' y selecciona muestras_kschool.zip")
    nombres = list(files.upload())

for nombre in nombres:
    try:
        with pyzipper.AESZipFile(nombre) as z:
            z.setpassword(b"infected")
            z.extractall("/content/")
    except Exception as e:
        print("✖ No he podido abrir", nombre, "->", e)

ficheros = sorted(f for f in glob.glob("/content/muestras/*") if not f.endswith("LEEME.txt"))
if ficheros:
    print("")
    print("✔ " + str(len(ficheros)) + " muestras listas en /content/muestras/")
    for f in ficheros:
        print("   {:>9,} bytes   {}".format(os.path.getsize(f), os.path.basename(f)))
else:
    print("✖ Sigo sin ver las muestras. Avisa en el chat.")

---
# Paso 1 · ¿De qué te fiarías?

Empecemos por lo que ve cualquier usuario: **el nombre del fichero**.

In [ ]:
import glob, os

ficheros = sorted(f for f in glob.glob("/content/muestras/*") if not f.endswith("LEEME.txt"))

print("Lo que vería un usuario en su correo:")
print()
for f in ficheros:
    print("   📄", os.path.basename(f))

Un usuario normal miraría esto y pensaría:

- `curriculum_vitae_2024.pdf` → *"es un PDF, un CV, lo abro"* ✅
- `informe_trimestral.pdf` → *"un PDF de contabilidad"* ✅
- `factura_adjunta.docx` → *"un Word, lo abro"* ✅
- `antivirus_update.exe` → *"bueno, es del antivirus"* 🤔

**Todo eso es información que ha escrito el atacante.** El nombre y la extensión de un fichero
son solo texto: cualquiera puede llamar `.pdf` a lo que quiera. Vamos a comprobarlo.

---
# Paso 2 · Los *magic bytes*: lo que el fichero es de verdad

Todo formato de fichero empieza por una **firma fija** en sus primeros bytes.
Es como el DNI del formato, y **lo pone el programa que lo creó**, no el atacante.

| Primeros bytes | En texto | Qué es de verdad |
|---|---|---|
| `4D 5A` | `MZ` | **Ejecutable de Windows** (.exe, .dll) |
| `25 50 44 46` | `%PDF` | Documento PDF |
| `50 4B 03 04` | `PK··` | Un ZIP — y también **.docx, .xlsx, .pptx** (por dentro son ZIPs) |
| `7F 45 4C 46` | `·ELF` | Ejecutable de Linux |
| `D0 CF 11 E0` | | Office **antiguo** (.doc, .xls de los de antes) |

> 💡 Las letras `MZ` son las iniciales de **Mark Zbikowski**, el ingeniero de Microsoft que
> diseñó el formato en 1981. Llevan ahí más de 40 años.

Ejecuta la celda: lee los **4 primeros bytes** de cada fichero y los compara con la tabla.

In [ ]:
import os

FIRMAS = {
    b"MZ":               "⚙️  EJECUTABLE DE WINDOWS (.exe / .dll)",
    b"%PDF":             "📕 Documento PDF",
    b"PK\x03\x04":       "🗜️  ZIP  (ojo: los .docx/.xlsx/.pptx también son ZIP)",
    b"\x7fELF":          "🐧 Ejecutable de Linux",
    b"\xd0\xcf\x11\xe0": "📘 Office antiguo (.doc / .xls)",
}

def que_es(ruta):
    """Lee los primeros bytes del fichero y los compara con la tabla de firmas."""
    with open(ruta, "rb") as f:
        cabecera = f.read(8)
    for firma, descripcion in FIRMAS.items():
        if cabecera.startswith(firma):
            return cabecera, descripcion
    return cabecera, "❓ Formato desconocido"

print("{:<42} {:<20} {}".format("FICHERO", "PRIMEROS BYTES", "QUÉ ES DE VERDAD"))
print("─" * 112)
for f in ficheros:
    cabecera, desc = que_es(f)
    hexa  = " ".join("{:02X}".format(b) for b in cabecera[:4])
    texto = "".join(chr(b) if 32 <= b < 127 else "." for b in cabecera[:4])
    print("{:<42} {}  {:<6} {}".format(os.path.basename(f), hexa, texto, desc))

---
## 🧩 TU TURNO (3 minutos)

En Linux existe un comando, **`file`**, que hace justo esto pero muchísimo mejor:
conoce miles de firmas y además te da detalles (si es de 32 o 64 bits, si es .NET, etc.).

En Colab, si una línea empieza por `!`, se ejecuta como un comando de Linux en vez de como Python.

**Escribe en la celda de abajo** (tal cual, con la exclamación delante):

```
!file /content/muestras/*
```

y pulsa ▶️. Luego compara su respuesta con la tabla de antes.

In [ ]:
# 👇 ESCRIBE AQUÍ TU COMANDO Y PULSA ▶️

---
# Paso 3 · El hash: el DNI del fichero

Ya sabemos que `curriculum_vitae_2024.pdf` es un ejecutable. Pero, **¿es malicioso?**

Para preguntárselo a alguien (a VirusTotal, a un antivirus, a otro analista) necesitamos
una forma de **nombrar ese fichero exacto** sin tener que mandárselo. Para eso está el **hash**.

Un hash es una función que convierte *cualquier* fichero en una cadena de longitud fija:

- Si el fichero cambia **un solo bit**, el hash cambia entero.
- Dos ficheros distintos, en la práctica, nunca dan el mismo hash.
- Es **de una sola dirección**: del hash no puedes reconstruir el fichero.

Por eso el hash es el idioma universal de la industria: cuando un informe de amenazas publica
un IOC (*Indicator of Compromise*), casi siempre es un SHA-256.

In [ ]:
import hashlib

def hashes(ruta):
    datos = open(ruta, "rb").read()
    return {
        "md5":    hashlib.md5(datos).hexdigest(),
        "sha1":   hashlib.sha1(datos).hexdigest(),
        "sha256": hashlib.sha256(datos).hexdigest(),
    }

objetivo = "/content/muestras/curriculum_vitae_2024.pdf"

for algoritmo, valor in hashes(objetivo).items():
    print("{:<8} {}".format(algoritmo.upper(), valor))

### Demuéstrate a ti mismo lo del "un solo bit"

Vamos a copiar el fichero y cambiarle **un byte**. Solo uno. Mira lo que le pasa al hash.

In [ ]:
import hashlib

datos = open(objetivo, "rb").read()

modificado = bytearray(datos)
modificado[5000] = (modificado[5000] + 1) % 256   # cambiamos UN byte de 95.232

print("Original :", hashlib.sha256(datos).hexdigest())
print("1 byte   :", hashlib.sha256(bytes(modificado)).hexdigest())
print()
print("Mismo tamaño:", len(datos) == len(modificado), "· Bytes distintos: 1 de", len(datos))
print()
print("👉 Y ESTE ES EL GRAN PROBLEMA DEL HASH:")
print("   para un antivirus que solo mire hashes, ese segundo fichero es COMPLETAMENTE nuevo.")
print("   El atacante solo tiene que recompilar o cambiar una coma para volverse invisible.")
print("   Por eso el análisis estático no se queda en el hash. Seguimos en el LAB 2.")

---
## 🧩 TU TURNO (4 minutos) — Pregúntale a VirusTotal

**VirusTotal** es un servicio de Google que guarda el análisis de miles de millones de ficheros.
Le puedes preguntar **por hash**, sin subir nada: si alguien en el mundo ya subió ese fichero,
te cuenta todo lo que sabe de él.

> ⚠️ Fíjate en el detalle importante: **buscar por hash no sube el fichero**.
> Eso importa mucho en la vida real: si subes un documento de tu empresa a VirusTotal,
> ese documento pasa a estar disponible para los clientes de pago de la plataforma.
> **Buscar hashes: siempre. Subir ficheros: solo si sabes qué estás subiendo.**

**Ejecuta la celda de abajo.** Te va a generar un enlace: ábrelo en otra pestaña.

In [ ]:
sha = hashes(objetivo)["sha256"]
print("SHA-256:", sha)
print()
print("👉 Abre este enlace en otra pestaña:")
print("   https://www.virustotal.com/gui/file/" + sha)
print()
print("Busca en la página y apunta:")
print("   a) Cuántos motores antivirus lo detectan (arriba a la izquierda, tipo 58/72)")
print("   b) Qué FAMILIA de malware nombran (mira las etiquetas y la columna de detecciones)")
print("   c) Pestaña DETAILS → ¿cuándo se compiló? ¿cuándo se vio por primera vez?")

### 📝 Apunta tus respuestas aquí

Haz doble clic sobre este texto para editarlo y escribe:

- Motores que lo detectan: `.......`
- Familia de malware: `.......`
- Fecha de compilación: `.......`

<br>

> **Pista de a dónde vamos:** esa familia es un **RAT** (*Remote Access Trojan*), un troyano de
> control remoto. En el LAB 2 vamos a sacarle, del propio fichero y sin ejecutarlo,
> **a qué servidor se conecta**.

---
# 🏆 RETO (para quien vaya sobrado)

> Si has llegado aquí y aún queda tiempo, sigue. Si no, **no pasa absolutamente nada**:
> esto lo vemos en la puesta en común y te lo llevas para casa.

## Reto A · ¿Cuál está empaquetado?

Un **packer** es un programa que comprime y cifra a otro programa. El malware los usa
constantemente porque así el código malicioso **no se ve** hasta que el programa se ejecuta.

¿Cómo lo detectamos sin ejecutar nada? Con la **entropía**.

La entropía mide *cuánto desorden* hay en los datos, de 0 a 8:

| Entropía | Qué significa |
|---|---|
| 0 – 3 | Muy repetitivo (ceros, texto plano) |
| 4 – 6 | Código de programa normal |
| **7,2 – 8** | **Comprimido o cifrado** → 🚩 sospechoso de estar empaquetado |

Un ejecutable normal ronda 5-6. Si te sale 7,9, ahí dentro no hay código: hay un bloque cifrado.

In [ ]:
import math, os
from collections import Counter

def entropia(datos):
    if not datos:
        return 0.0
    n = len(datos)
    return -sum((c/n) * math.log2(c/n) for c in Counter(datos).values())

print("{:<42} {:>9}   VEREDICTO".format("FICHERO", "ENTROPÍA"))
print("─" * 108)
for f in ficheros:
    e = entropia(open(f, "rb").read())
    if e >= 7.2:
        veredicto = "🚩 COMPRIMIDO/CIFRADO — sospechoso de estar empaquetado"
    elif e >= 6.0:
        veredicto = "⚠️  tiene partes comprimidas (normal en PDF/DOCX o con muchos recursos)"
    else:
        veredicto = "✅ parece código normal"
    print("{:<42} {:>8.2f}   {}".format(os.path.basename(f), e, veredicto))
    print("{:<42} {}".format("", "█" * int(e * 5)))

### Reto B · La prueba definitiva: mirar las secciones del ejecutable

La entropía te da una sospecha. Para confirmarla abrimos el ejecutable por dentro.

Un ejecutable de Windows (formato **PE**) está dividido en **secciones**, y sus nombres son
bastante estándar: `.text` (el código), `.data` (las variables), `.rsrc` (iconos, imágenes…).

Cuando un packer procesa un fichero, **deja su firma en los nombres de las secciones**.
UPX, el packer más común del mundo, las llama `UPX0` y `UPX1`.

De paso miramos el **imphash**: un hash calculado solo sobre la lista de funciones que el
programa importa de Windows. Dos muestras con el mismo imphash suelen venir del mismo compilador
o del mismo autor — sirve para **agrupar familias**, no para identificar ficheros concretos.

In [ ]:
# Dependencias de esta celda (por si la ejecutas sin pasar por la de arriba)
import importlib, subprocess, sys
for _m, _p in [("pefile", "pefile")]:
    try:
        importlib.import_module(_m)
    except ImportError:
        print('Instalando ' + _p + '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p],
                       check=False)

import pefile

import os

def radiografia(ruta):
    pe = pefile.PE(ruta)
    print("┌─", os.path.basename(ruta))
    print("│  imphash:", pe.get_imphash())
    print("│  secciones:")
    for s in pe.sections:
        nombre = s.Name.decode(errors="replace").rstrip("\x00")
        marca  = "   🚩 ¡FIRMA DE UPX!" if nombre.upper().startswith("UPX") else ""
        vacia  = "   ← ocupa 0 en disco: se rellena al ejecutarse" if s.SizeOfRawData == 0 else ""
        print("│     {:<10} entropía={:.2f}  disco={:>9,}{}{}".format(
            nombre, s.get_entropy(), s.SizeOfRawData, marca, vacia))
    dirs = getattr(pe, "DIRECTORY_ENTRY_IMPORT", [])
    print("│  importa {} funciones de {} librerías de Windows".format(
        sum(len(d.imports) for d in dirs), len(dirs)))
    print("└─")
    print()

radiografia("/content/muestras/actualizacion_flash.exe")
radiografia("/content/muestras/actualizacion_flash_DESEMPAQUETADO.bin")

### 🔬 Compara las dos radiografías de arriba

Son **el mismo programa**: el de abajo es el de arriba después de desempaquetarlo.

| | Empaquetado | Desempaquetado |
|---|---|---|
| Secciones | `UPX0`, `UPX1`, `.rsrc` | `.text`, `.rdata`, `.data`, `.rsrc`, `.reloc` |
| Entropía de la sección grande | **7,94** 🚩 | 6,68 |
| `UPX0` en disco | **0 bytes** (!) | — |
| Funciones importadas | **~24** | **~500** |
| imphash | `fc6683d3…` | `dcbf4f6f…` |

**Las dos pistas que más te van a servir en la vida real:**

1. **Una sección que ocupa 0 bytes en disco pero reserva 1 MB en memoria.** Eso es el packer
   diciendo *"aquí voy a descomprimir el programa de verdad cuando me ejecuten"*.
2. **Un programa que solo importa 24 funciones.** Un troyano de control remoto necesita red,
   ficheros, registro, teclado… cientos de funciones. Si solo importa
   `LoadLibraryA`, `GetProcAddress`, `VirtualAlloc` y `VirtualProtect`, lo que tienes delante
   **no es el programa: es el desempaquetador**. Le va a pedir memoria a Windows, escribir ahí
   el programa real y saltar a él.

> Y fíjate en el **imphash**: cambia entre las dos versiones. Por eso el imphash agrupa
> *"malware empaquetado con UPX de esta manera"*, que también es información útil para cazar.

### Reto C · Los metadatos también mienten

Última parada. Abre `antivirus_update.exe` y mira quién dice ser.

In [ ]:
# Dependencias de esta celda (por si la ejecutas sin pasar por la de arriba)
import importlib, subprocess, sys
for _m, _p in [("pefile", "pefile")]:
    try:
        importlib.import_module(_m)
    except ImportError:
        print('Instalando ' + _p + '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p],
                       check=False)

import pefile, os

pe = pefile.PE("/content/muestras/antivirus_update.exe")

print("Lo que este fichero dice de sí mismo:")
print()
for fileinfo in (pe.FileInfo or []):
    for entry in fileinfo:
        if hasattr(entry, "StringTable"):
            for st in entry.StringTable:
                for k, v in st.entries.items():
                    print("   {:<18} {}".format(k.decode(errors="replace"),
                                                v.decode(errors="replace")))
print()
print("Secciones:", [s.Name.decode(errors="replace").rstrip("\x00") for s in pe.sections])

### 🤨 ¿Te lo crees?

Dice ser de **"AVG Technologies CZ, s.r.o."**. Un antivirus. Pero:

- El nombre del producto es **`barnumism`** y la descripción **`frownful`**: palabras de
  diccionario al azar, generadas por un script. Ningún producto real se llama así.
- Sus secciones son `UPX0` / `UPX1`: está empaquetado.
- El copyright es `conservant`, que tampoco significa nada.

**Moraleja del LAB 1:** todo lo que un fichero *dice* de sí mismo — su nombre, su extensión,
su icono, su empresa, su copyright — **lo ha escrito quien lo creó**. Lo único que no puede
falsificar con facilidad es **su estructura**: magic bytes, secciones, entropía, imports.
Ahí es donde mira el analista.

---
# ✅ Antes de la puesta en común

Ten a mano estas tres respuestas:

1. 🏷️ El fichero que miente: **`________________`** — dice ser `.____` y en realidad es `____`
2. 🔢 Su SHA-256 empieza por **`________`** y VirusTotal dice que es la familia **`________`**
3. 🎁 Está empaquetado: **`________________`**, y lo sé porque **`________________`**

> Si no te ha dado tiempo al RETO, **tranquilo**: era opcional y lo vemos juntos ahora.

**Siguiente:** LAB 2 · *Encuentra el C2* — le vamos a sacar al troyano la dirección de su
servidor de mando y control. Sin ejecutarlo.